# Filtered and Incremental ANN — Predicate Search, Deletion, and Graph Connectivity

The narrative companion to `filtered_incremental_ann.py`, which **owns every number**. HNSW gave us a
graph that is natively incremental on *insert* and a beam search that walks it. Production indexes
face two things the static graph ignores: vectors are **deleted** over time, and queries carry a
**predicate** the index must respect. The unifying idea is that both are the same operation —
removing nodes from a navigable graph — and the mathematics has three movements, with the **over-fetch
laws** as the exact spine and **percolation** as an honestly-bounded floor. We walk all three and call
the verification harness; each assertion encodes a claim the topic makes.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "filtered-incremental-ann",
              pathlib.Path("notebooks/filtered-incremental-ann")):
    if (_cand / "filtered_incremental_ann.py").exists():
        sys.path.insert(0, str(_cand)); break
import numpy as np
import filtered_incremental_ann as fia
from navigable_small_world_graphs import nsw_dataset
X, queries = nsw_dataset()
print(f"cloud: {X.shape[0]} vectors in R^{X.shape[1]}, {len(queries)} held-out queries")

## Movement 1 — insertion is free; deletion is the hard half (the exact spine)

HNSW inserts one node at a time, so growth is free; the asymmetry is **deletion**. A *tombstoned*
node stays in the graph as a routing waypoint but is dropped from results. To return $k$ **live**
results when a fraction $\delta$ of nodes are dead, the number of candidates scanned is
negative-binomial:
$$\mathbb{E}[N_k] = \frac{k}{1-\delta}, \qquad \operatorname{Var}(N_k) = \frac{k\,\delta}{(1-\delta)^2}.$$
Exact under one idealization: that live/dead is independent of position in the ranked stream.

In [ ]:
fia.test_overfetch_law_tombstone()
for r in fia.measure_overfetch_law(X, queries, fia.DELTA_GRID, topk=fia.TOPK, trials=40):
    print(f"  delta={r['delta']:.2f}: scanned {r['scanned']:.2f}  vs  k/(1-delta) {r['predicted']:.2f}"
          f"   (var {r['var']:.1f} vs {r['var_pred']:.1f})")

Tombstones accumulate: dead nodes still cost a distance computation during traversal and crowd
the beam, so recall sags. **Hard deletion + a neighbor-repair heuristic** — excise the node and
re-link each orphaned neighbor to preserve out-degree $M$ — recovers it. The repair has no
optimality proof; we show only that it restores what tombstone bloat costs.

In [ ]:
fia.test_hard_delete_repair_restores_recall()
r = fia.recall_after_repair(X, queries, delta=0.3, ef=32, seed=0)
print(f"  deleted {r['n_deleted']} of {X.shape[0]};  recall tombstoned {r['rec_tombstoned']:.3f}"
      f"  ->  repaired {r['rec_repaired']:.3f};  layer-0 now {r['layer0_after']} = {r['n_remaining']} survivors")

## Movement 2 — connectivity under churn = percolation (the honest floor)

For a random graph in which every node has degree exactly $M$, retaining each node with probability
$p$ leaves a giant connected component **iff**
$$p > p_c = \frac{1}{M-1}$$
(Molloy–Reed / Cohen et al.; the excess-degree branching ratio is $M-1$). Note this is the
**regular-graph** value — *not* the Erdős–Rényi $1/M$ ($\kappa=M$ vs $\kappa=M+1$). We verify it on a
near-regular configuration-model graph, where the theorem is exact.

In [ ]:
fia.test_regular_graph_percolation_threshold()
reg = fia.random_regular_graph(fia.REG_N, fia.REG_DEG, seed=0)
s = fia.degree_stats(reg)
print(f"  near-regular graph: <k>={s['mean_deg']:.2f}, kappa={s['kappa']:.2f}, "
      f"p_c=1/(kappa-1)={s['pc']:.3f}")

On HNSW's **real** layer-0 graph the threshold only *approximates* the configuration-model law
— degree correlations and metric structure shift it. And the load-bearing caveat: **connectivity is
necessary but not sufficient for navigability.** Greedy search returns a local minimum, so recall
fails far *inside* the connected regime — the percolation threshold is never the binding constraint.
A denser graph (larger $M$) is the lever that buys churn robustness.

In [ ]:
fia.test_giant_component_monotone_in_M()
for r in fia.connectivity_vs_M((4, 8, 16, 32), X, retain_p=0.15, trials=10, seed=0):
    print(f"  M={r['M']:>2}  <k>={r['mean_deg']:.2f}  giant fraction @retain 0.15 = {r['giant_frac']:.3f}")

## Movement 3 — predicate search = query-time deletion (the unification + the headline)

A filter soft-deletes every failing node **for this query**: a tombstone is a persistent global
predicate, a predicate is a per-query tombstone. So post-filtering obeys the *same* law — mean fetch
$k/s$ for selectivity $s$ — with a sharp binomial recall cliff once $s < k/F$ under a fetch cap $F$.

First the correctness anchor: the in-filter beam `search_layer_filtered` must reduce to HNSW's
`search_layer` exactly when every node is live (same ids **and** distance count).

In [ ]:
fia.test_search_layer_filtered_matches_unfiltered_when_all_live()

The post-filter over-fetch law and its recall cliff, isolated on the exact ranked stream:

In [ ]:
fia.test_postfilter_overfetch_law()
fia.test_postfilter_recall_cliff()
for r in fia.postfilter_overfetch_law(X, queries, fia.S_GRID, topk=fia.TOPK, trials=40):
    print(f"  s={r['s']:.2f}: scanned {r['scanned']:.1f}  vs  k/s {r['predicted']:.1f}")

**The headline crossover (measured on this cloud — no universal winner).** Three strategies
trade off with selectivity: *pre-filter* (brute-force the passing subset) is exact and cheapest at
**low** $s$ — a tiny subset to scan; *post-filter* (search then drop) has constant cost but falls off
the cliff at low $s$; *in-filter* (traverse through failing nodes, collect only passing) stays exact
but at very low $s$ **degenerates toward a full scan** to find $k$ passing results. This is exactly
why filtered-ANN systems pre-filter below a selectivity threshold or build denser, predicate-aware
graphs.

In [ ]:
fia.test_filter_strategy_crossover()
layers, entry, top = fia.build_hnsw(X, M=fia.M_DEF, ef_construction=fia.EFC_DEF, seed=0)
fr = fia.filter_strategy_frontier(layers, X, queries, entry, top, fia.S_GRID, ef=64, topk=fia.TOPK, seed=0)
print(f"  {'s':>6} {'pre rec/cost':>16} {'post rec/cost':>16} {'in rec/cost':>16}")
for a, b, c in zip(fr['pre'], fr['post'], fr['in']):
    print(f"  {a['s']:>6.3f} {a['recall']:.2f}/{a['ndist']:>7.1f}     "
          f"{b['recall']:.2f}/{b['ndist']:>7.1f}     {c['recall']:.2f}/{c['ndist']:>7.1f}")

The percolation link returns, and the direction is the one the simulation reveals — not the
one intuition guesses. The induced subgraph on a **random** passing set is site percolation, so it
fragments once $s$ falls below $p_c$ (its giant fraction collapses). A spatially **coherent**
modality predicate (a contiguous cluster of the embedding space) stays connected far below that. So
predicate hardness depends on **spatial coherence**, not selectivity alone.

In [ ]:
fia.test_predicate_subgraph_percolation()
labels = fia.assign_modalities(X, n_modalities=5, seed=0)
for r in fia.predicate_subgraph_connectivity(layers[0], labels, trials=12, seed=0):
    print(f"  s={r['s']:.3f}: induced-subgraph giant fraction  random {r['giant_random']:.3f}"
          f"   vs  coherent/modality {r['giant_corr']:.3f}")

## The numbers the laboratory mirrors

`viz_constants()` prints the toy graph and every law, sweep, and frontier that
`FilteredIncrementalANNLaboratory.tsx` mirrors to the decimal.

In [ ]:
fia.viz_constants()
print("All checks passed.")